# Week 7a: A neural network from scratch (NumPy only)

You'll build a 2-layer network by hand, the same job as your Week 2 categorizer (description → category), and **prove your backpropagation is right** with a gradient check. No PyTorch yet: every number is yours.

Network: TF-IDF features → **64 hidden ReLU units** → **11 outputs** (softmax).

Spec: `docs/superpowers/specs/2026-10-23-deep-learning-design.md`

## Setup (given)
Your Week 2 data, cleaning, split and TF-IDF, turned into plain NumPy arrays. `Y_train` is one-hot: one column per category, a 1 in the right one.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

sys.path.insert(0, '../service')
from clean import clean_description   # the same cleaning the deployed model uses

df = pd.read_csv('../data/fake_transactions.csv')
df['Clean'] = df['Description'].fillna('').str.strip().apply(clean_description)
data = df[df['Clean'] != '']
text_train, text_test, cat_train, cat_test = train_test_split(
    data['Clean'], data['Category'], test_size=0.2, stratify=data['Category'], random_state=42)

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)).fit(text_train)
X_train, X_test = vec.transform(text_train).toarray(), vec.transform(text_test).toarray()   # float64
LABELS = sorted(data['Category'].unique())
y_train = np.array([LABELS.index(c) for c in cat_train])
y_test = np.array([LABELS.index(c) for c in cat_test])
Y_train = np.eye(len(LABELS))[y_train]

N_IN, N_HIDDEN, N_OUT = X_train.shape[1], 64, len(LABELS)
print(X_train.shape, X_test.shape, N_OUT, 'categories')

---
## A. Neurons, layers and activations
A layer is `X @ W + b`: every output is a weighted sum of the inputs plus a bias.

Write two functions:
- `relu(z)`: `max(0, z)` element-wise
- `softmax(z)`: for each row, `exp(z) / sum(exp(z))`, so the row adds up to 1. Subtract the row's max first (`z - z.max(axis=1, keepdims=True)`): the result is the same, but `exp` can't overflow.

**Think:** why do we need ReLU at all? What would two layers `(X @ W1) @ W2` be without it? (Hint: multiply `W1 @ W2` first.)

In [ ]:
# TODO


---
## B. Loss
`cross_entropy(probs, Y)` = the mean over rows of `−Σ Y · log(probs)`. Add `1e-12` inside the log so it never sees 0.

**Think:** we report accuracy, but we **train** on cross-entropy. Why can't gradient descent use accuracy directly? (Hint: what happens to accuracy when a weight changes by a tiny amount?)

In [ ]:
# TODO


---
## C. Forward pass
- `init_params()`: a dict with `W1 (N_IN×N_HIDDEN)`, `b1`, `W2 (N_HIDDEN×N_OUT)`, `b2`. Use `rng = np.random.default_rng(0)`, weights `rng.standard_normal(shape) * np.sqrt(2 / fan_in)` (draw `W1` first, then `W2`), biases zeros. Notebook 08 rebuilds exactly these weights, so keep this order.
- `forward(params, X)`: `Z1 = X @ W1 + b1`, `A1 = relu(Z1)`, `Z2 = A1 @ W2 + b2`, `probs = softmax(Z2)`. Return `probs` and a `cache` dict with `X, Z1, A1, probs` (backprop needs them).

Before training, the loss should be close to `log(11) ≈ 2.4`. Why?

In [ ]:
# TODO


---
## D. Backpropagation by hand
`backward(params, cache, Y)` returns a dict of gradients with the **same keys and shapes** as `params`.

Work backwards with the chain rule. The first step is given, because softmax + cross-entropy simplify beautifully:

`dZ2 = (probs − Y) / n`  (n = rows in the batch)

Then:
- `dW2`: how does the loss change with `W2`? (`Z2 = A1 @ W2`, so it involves `A1.T`)
- `db2`: sum `dZ2` over rows
- `dA1`: pass the gradient back through `W2`
- `dZ1`: ReLU lets the gradient through only where `Z1 > 0`
- `dW1`, `db1`: the same pattern as layer 2

**Tip:** check every shape. `dW2` must be `(64, 11)` like `W2`.

In [ ]:
# TODO


---
## E. Gradient check (given)
The proof your maths is right. For a few weights, nudge the weight by ±ε, measure the loss, and compare `(L(w+ε) − L(w−ε)) / 2ε` with your gradient. If the relative error is below 1e-6, your backprop is correct.

In [ ]:
def gradient_check(params, X, Y, eps=1e-5, per_param=20, seed=1):
    rng = np.random.default_rng(seed)
    grads = backward(params, forward(params, X)[1], Y)
    worst = 0.0
    for name, P in params.items():
        G = grads[name]
        assert G.shape == P.shape, f'{name}: gradient shape {G.shape} vs {P.shape}'
        assert np.abs(G).max() > 1e-8, f'{name}: gradient is all ~0'
        # weights whose gradient is too tiny to measure numerically (e.g. unused inputs) can't be checked
        candidates = np.argwhere(np.abs(G) > 1e-6)
        for idx in map(tuple, candidates[rng.choice(len(candidates), min(per_param, len(candidates)), replace=False)]):
            old = P[idx]
            P[idx] = old + eps; plus = cross_entropy(forward(params, X)[0], Y)
            P[idx] = old - eps; minus = cross_entropy(forward(params, X)[0], Y)
            P[idx] = old
            numeric = (plus - minus) / (2 * eps)
            worst = max(worst, abs(G[idx] - numeric) / max(abs(G[idx]) + abs(numeric), 1e-12))
    return worst

worst = gradient_check(init_params(), X_train[:32], Y_train[:32])
print(f'worst relative error: {worst:.2e}')
assert worst < 1e-6, 'backprop has a bug: compare each gradient with its numeric value'
print('backprop ✓')

---
## F. Train it
Mini-batch **SGD**: shuffle the rows each epoch, then for each batch of 64: forward, backward, and `param -= LR * grad` for every parameter.

Start with `LR = 0.5`, 30 epochs. After each epoch record the training loss, train accuracy and test accuracy in a list of dicts, then `history = pd.DataFrame(...).set_index('epoch')`. The `accuracy` helper is given.

In [ ]:
def accuracy(params, X, y):
    return float((forward(params, X)[0].argmax(axis=1) == y).mean())

In [ ]:
# TODO


In [ ]:
# Plot (given)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
history['loss'].plot(ax=ax[0], title='Training loss')
history[['train_acc', 'test_acc']].plot(ax=ax[1], title='Accuracy', ylim=(0, 1))
plt.show()
print(f"test accuracy: {history['test_acc'].iloc[-1]:.1%}  (Week 2 scikit-learn: 96.3%)")

**Think:** re-run training with `LR = 5` and with `LR = 0.05` (keep the rest). What happens to the loss curve in each case, and why?

**Your answer:** TODO